# z719 final gerencial v9: validación a dos meses de distancia (WF7199)
Francisco Medina Raed. **Objetivo:** decidir el corte de la entrega con datos.

Julio siempre marcó el óptimo cerca de 800 envíos y el Public cerca de 1.100. Una explicación posible es la **distancia**: la entrega entrena hasta 202107 y predice 202109 (dos meses), mientras que el grid y las curvas de julio entrenaban hasta 202106 y medían en 202107 (un mes).

Este notebook reproduce las dos distancias, con el modelo del WF7197/WF7198 (learning rate 0,02, 256 hojas, mínimo 128, 150 árboles) y **5 meses de training**, como el final_train de la entrega:

| Escenario | Training | Prueba | Distancia |
|---|---|---|---|
| prueba202107_dist1 | 202102 a 202106 | 202107 | 1 mes |
| prueba202107_dist2 | 202101 a 202105 | 202107 | **2 meses**, como la entrega |
| prueba202106_dist1 | 202101 a 202105 | 202106 | 1 mes |
| prueba202106_dist2 | 202012 a 202104 | 202106 | **2 meses** |

Cada escenario es un semillerío de 20. Medir en dos meses de prueba (175 BAJA+2 en total) da más casos que solo julio.

**Qué muestra**
- Para cada escenario: el corte óptimo, la meseta (envíos a menos de 1 M del óptimo), la ganancia en 800 a 1.200 envíos y cuántos BAJA+2 encuentra.
- **Promediando los dos meses de prueba**, para cada distancia: el óptimo y cuánto se pierde cortando en 800, 900, 1.000, 1.100 o 1.200.
- Un gráfico con las curvas a uno y dos meses.

**Cómo leerlo:** si a dos meses el óptimo se corre hacia 1.000–1.100, la distancia explica la diferencia con el Public y el corte de 1.100 queda respaldado. Si se queda cerca de 800, conviene cortar más abajo.

No sube nada a Kaggle. Son 80 modelos; calculo unos 10 minutos. Es reanudable. Primero conviene `modo <- "test"`.

## Parámetros, paquetes y carpetas

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

modo <- "full"           # "test" = corrida corta para probar que todo anda (no sube a Kaggle)
subir_kaggle <- FALSE    # este notebook solo analiza: no sube nada a Kaggle

PARAM <- list()
PARAM$semilla_primigenia <- 123457     # <- tu semilla primigenia
PARAM$experimento <- 7199
PARAM$comparar_con <- c(7190, 7191, 7192, 7193, 7194, 7195, 7196, 7197, 7198)           # experimentos anteriores que se muestran en la curva de puntajes
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# --- Feature engineering: cada bloque se prende o se apaga por separado ---
# Cada combinacion de bloques se guarda en su propia carpeta y con su propio nombre en Kaggle,
# asi se pueden correr de a uno y comparar sin pisarse.
PARAM$fe$sacar_ids  <- TRUE     # numero_de_cliente, foto_mes y kmes dejan de ser variables del modelo
PARAM$fe$ratios     <- TRUE     # bloque 2: ratios y sumas dentro del mes (senales de fuga)
PARAM$fe$rankings   <- TRUE     # bloque 1: montos convertidos a ranking por mes (data drifting)
PARAM$fe$tendencias <- TRUE     # bloque 3: tendencia, promedio, max, min y ratio en ventana, sobre las top variables
PARAM$fe$tendencias_top     <- 40
PARAM$fe$tendencias_ventana <- 6
PARAM$fe$ratios2    <- FALSE    # NUEVO: segunda tanda de ratios, con columnas que si tiene el dataset gerencial
PARAM$fe$canaritos  <- FALSE    # NUEVO: descarta las variables que el modelo usa menos que variables de puro azar
PARAM$fe$canaritos_cantidad    <- 50     # cuantas variables de azar se agregan
PARAM$fe$canaritos_semillas    <- 5      # modelos con los que se promedia la importancia
PARAM$fe$canaritos_iteraciones <- 200    # arboles de cada uno de esos modelos
PARAM$fe$canaritos_umbral      <- 0.5    # una variable queda si supera a esta fraccion de los canaritos (0.5 = la mediana)

etiqueta <- paste(c(if (PARAM$fe$sacar_ids) "noid", if (PARAM$fe$rankings) "rank",
                    if (PARAM$fe$ratios) "ratio", if (PARAM$fe$ratios2) "ratio2",
                    if (PARAM$fe$tendencias) "tend", if (PARAM$fe$canaritos) "can"), collapse = "_")
if (etiqueta == "") etiqueta <- "base"

# --- Training strategy: la ganadora del Exp 07, del WF7190 y del WF7191 ---
PARAM$variantes <- list(
  list(nombre = paste0("V3fe_", etiqueta, "_val2m"), desde = 202103,
       descripcion = paste0("training 202103-202106, final_train 202103-202107, grid por ganancia, FE: ", etiqueta))
)
PARAM$trainingstrategy$validate     <- 202107
PARAM$trainingstrategy$hasta        <- 202106   # ultimo mes de training
PARAM$trainingstrategy$future       <- 202109
PARAM$trainingstrategy$training_pct <- 1.0      # sin undersampling, como el z719
PARAM$trainingstrategy$positivos    <- c("BAJA+1", "BAJA+2")

# --- Ganancia de la competencia ---
PARAM$ganancia$acierto <- 975000    # por cada BAJA+2 contactado
PARAM$ganancia$error   <- -25000    # por cada BAJA+1 o CONTINUA contactado
PARAM$ganancia$ventana <- 401       # suavizado de la curva de ganancia en validacion (clientes)

# --- Grid search: ahora se elige por GANANCIA en 202107, no por AUC ---
# min_data_in_leaf arranca en 64: con 16 y 32 el WF7190 sobreajusto (V2 se desplomo en Kaggle)
PARAM$grid <- list(
  num_leaves       = c(64, 128, 256, 512, 1024),
  min_data_in_leaf = c(64, 128, 256, 512, 1024, 2048)
)
PARAM$grid$semillas_eval <- 10      # cada combinacion se evalua con 10 semillas y se promedia (la ganancia es ruidosa)

# --- Parametros fijos de LightGBM: identicos al z719 ---
PARAM$lgbm$param_fijos <- list(
  objective = "binary",
  metric = "None",              # la metrica es la ganancia (feval), no el AUC
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,
  max_bin = 31,
  learning_rate = 0.03,
  feature_fraction = 0.5,
  num_iterations = 2048,        # lo limita el early stopping
  early_stopping_rounds = 200,
  num_leaves = 64,
  min_data_in_leaf = 128
)

# --- Hiperparametros del modelo: los del WF7197 / WF7198 (learning_rate 0.02). No se hace grid search. ---
PARAM$modelo <- list(num_leaves = 256, min_data_in_leaf = 128, learning_rate = 0.02, num_iterations = 150)

# --- Validacion a dos meses ---
# La entrega entrena con 5 meses que terminan en 202107 y predice 202109: DOS meses de distancia.
# El grid y las curvas de julio entrenaban hasta 202106 y median en 202107: UN mes de distancia.
# Aca se reproducen las dos distancias, con 5 meses de training en ambas, en dos meses de prueba.
PARAM$validacion$meses_prueba <- c(202107, 202106)   # meses con clase donde se mide
PARAM$validacion$distancias   <- c(1, 2)             # meses entre el ultimo de training y el de prueba
PARAM$validacion$meses_train  <- 5                   # cantidad de meses de training (como el final_train de la entrega)
PARAM$validacion$semillas     <- 20                  # semillerio de cada escenario
PARAM$validacion$candidatos   <- c(800, 900, 1000, 1100, 1200)   # cortes a comparar

# --- Semillerio (de aca se toman las semillas de la validacion) ---
PARAM$semillerio$cantidad <- 20

# --- Opcional: escalar num_iterations por las filas que agrega 202107 al final_train ---
PARAM$final$escalar_iteraciones <- FALSE

# --- Kaggle: cortes de a 50 alrededor del pico del Exp 07 (unos 1.100 envios) ---
PARAM$kaggle$competencia <- "labo-1-virtual-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1400, by = 50)

if (modo == "test") {
  PARAM$experimento <- paste0(PARAM$experimento, "_test")
  PARAM$grid <- list(num_leaves = c(64, 256), min_data_in_leaf = c(64, 512), semillas_eval = 2)
  PARAM$modelo$num_iterations <- 40
  PARAM$validacion$semillas <- 4
  PARAM$lgbm$param_fijos$num_iterations <- 60
  PARAM$lgbm$param_fijos$early_stopping_rounds <- 20
  PARAM$semillerio$cantidad <- 8
  subir_kaggle <- FALSE
}

# semillas del semillerio: la primera es la primigenia, el resto se derivan de ella (reproducible)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillerio$semillas <- c(PARAM$semilla_primigenia,
  setdiff(sample(100000:999999, PARAM$semillerio$cantidad + 5), PARAM$semilla_primigenia)[seq_len(PARAM$semillerio$cantidad - 1)])

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

# carpeta del experimento en el bucket de la VM (o en el home si no hay bucket)
base <- path.expand(if (dir.exists("~/buckets/b1")) "~/buckets/b1/exp" else "~/exp")
carpeta_exp <- file.path(base, paste0("WF", PARAM$experimento))
dir.create(carpeta_exp, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_exp)

# dataset: lo busco donde lo suele dejar la VM; si no esta, lo descargo
candidatos <- path.expand(file.path(c("~/datasets", "~/buckets/b1/datasets", carpeta_exp), PARAM$dataset))
archivo_dataset <- candidatos[file.exists(candidatos)][1]
if (is.na(archivo_dataset)) {
  archivo_dataset <- file.path(carpeta_exp, PARAM$dataset)
  download.file(paste0("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/", PARAM$dataset),
                archivo_dataset, mode = "wb")
}

message("variante: ", PARAM$variantes[[1]]$nombre, "\ncarpeta: ", carpeta_exp, "\ndataset: ", archivo_dataset,
        "\nsemillas del semillerio: ", paste(PARAM$semillerio$semillas, collapse = ", "))

## Preprocesamiento y feature engineering
Igual que el WF7193.

In [ ]:
dataset <- fread(archivo_dataset)

# ---------- 6.3.1.2 Catastrophe Analysis (identico al z719) ----------
dataset[foto_mes == 202006, internet := NA]
dataset[foto_mes == 202006, mrentabilidad := NA]
dataset[foto_mes == 202006, mrentabilidad_annual := NA]
dataset[foto_mes == 202006, mcomisiones := NA]
dataset[foto_mes == 202006, mactivos_margen := NA]
dataset[foto_mes == 202006, mpasivos_margen := NA]
dataset[foto_mes == 202006, mcuentas_saldo := NA]
dataset[foto_mes == 202006, ctarjeta_visa_transacciones := NA]
dataset[foto_mes == 202006, mtarjeta_visa_consumo := NA]
dataset[foto_mes == 202006, mtarjeta_master_consumo := NA]
dataset[foto_mes == 202006, ccallcenter_transacciones := NA]
dataset[foto_mes == 202006, chomebanking_transacciones := NA]

# ---------- 6.3.1.3 FE intra-mes (identico al z719) ----------
atributos_presentes <- function(patributos) {
  atributos <- unique(patributos)
  length(atributos) == length(intersect(atributos, colnames(dataset)))
}
if (atributos_presentes(c("foto_mes")))
  dataset[, kmes := foto_mes %% 100]
if (atributos_presentes(c("mpayroll", "cliente_edad")))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# ---------- BLOQUE 2: ratios y sumas dentro del mes ----------
# Se calculan ANTES de los rankings: un cociente entre dos montos del mismo mes no tiene inflacion.
# Las sumas de montos se llaman m... (despues se rankean); los cocientes se llaman r_... (no se rankean).
fe_creadas <- character(); fe_salteadas <- character()
# suma por fila que da NA solo si todos los sumandos son NA (no revive los ceros de 202006)
suma_na <- function(...) {
  m <- cbind(...)
  s <- rowSums(m, na.rm = TRUE)
  s[rowSums(!is.na(m)) == 0] <- NA
  s
}
# crea la variable solo si estan todas las columnas que necesita; si no, la saltea y avisa
agregar <- function(nombre, requeridos, expr) {
  if (atributos_presentes(requeridos)) {
    dataset[, (nombre) := eval(expr)]
    fe_creadas <<- c(fe_creadas, nombre)
  } else {
    fe_salteadas <<- c(fe_salteadas, paste0(nombre, " (falta: ",
      paste(setdiff(requeridos, colnames(dataset)), collapse = ", "), ")"))
  }
}

if (PARAM$fe$ratios) {
  # actividad total con tarjetas de credito
  agregar("mtarjetas_consumo", c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"),
          quote(suma_na(mtarjeta_visa_consumo, mtarjeta_master_consumo)))
  agregar("ctarjetas_transacciones", c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones"),
          quote(suma_na(ctarjeta_visa_transacciones, ctarjeta_master_transacciones)))
  # cuanto del limite de la tarjeta esta usando
  agregar("r_visa_uso_limite", c("Visa_msaldototal", "Visa_mlimitecompra"),
          quote(Visa_msaldototal / Visa_mlimitecompra))
  agregar("r_master_uso_limite", c("Master_msaldototal", "Master_mlimitecompra"),
          quote(Master_msaldototal / Master_mlimitecompra))
  # alguna tarjeta en proceso de cierre o cerrada (status 6, 7 o 9)
  agregar("tarjeta_cerrando", c("Visa_status", "Master_status"),
          quote(as.integer(Visa_status %in% c(6, 7, 9) | Master_status %in% c(6, 7, 9))))
  # que parte de la operatoria es por homebanking
  agregar("r_digital", c("chomebanking_transacciones", "ccallcenter_transacciones", "ccajas_transacciones"),
          quote(chomebanking_transacciones /
                (chomebanking_transacciones + ccallcenter_transacciones + ccajas_transacciones)))
  # cobra el sueldo en el banco; su delta1 = -1 es "dejo de cobrar el sueldo aca"
  agregar("tiene_payroll", c("mpayroll"), quote(as.integer(mpayroll > 0)))
  # saldo y consumo medidos en sueldos
  agregar("r_saldo_sobre_payroll", c("mcuentas_saldo", "mpayroll"),
          quote(ifelse(mpayroll > 0, mcuentas_saldo / mpayroll, NA_real_)))
  agregar("r_consumo_sobre_payroll", c("mtarjetas_consumo", "mpayroll"),
          quote(ifelse(mpayroll > 0, mtarjetas_consumo / mpayroll, NA_real_)))
  # intensidad de uso: transacciones del trimestre por producto contratado
  agregar("r_trx_por_producto", c("ctrx_quarter", "cproductos"), quote(ctrx_quarter / cproductos))
}

# ---------- BLOQUE 2 bis: segunda tanda de ratios, con columnas del dataset gerencial ----------
if (PARAM$fe$ratios2) {
  # endeudamiento: prestamos personales medidos en sueldos
  agregar("r_prestamos_sobre_payroll", c("mprestamos_personales", "mpayroll"),
          quote(ifelse(mpayroll > 0, mprestamos_personales / mpayroll, NA_real_)))
  # que parte de la operatoria es por homebanking (reemplaza a r_digital, que no se puede crear)
  agregar("r_homebanking_sobre_trx", c("chomebanking_transacciones", "ctrx_quarter"),
          quote(chomebanking_transacciones / ctrx_quarter))
  # llamados al call center en relacion a la operatoria: reclamos o gestiones de baja
  agregar("r_callcenter_sobre_trx", c("ccallcenter_transacciones", "ctrx_quarter"),
          quote(ccallcenter_transacciones / ctrx_quarter))
  # donde tiene la plata: que parte del saldo esta en caja de ahorro
  agregar("r_caja_ahorro_sobre_saldo", c("mcaja_ahorro", "mcuentas_saldo"),
          quote(ifelse(mcuentas_saldo > 0, mcaja_ahorro / mcuentas_saldo, NA_real_)))
  # cuanto de lo que el banco gana con el cliente son comisiones
  agregar("r_comisiones_sobre_rentabilidad", c("mcomisiones", "mrentabilidad"),
          quote(mcomisiones / mrentabilidad))
  # margen total que deja el cliente (activos + pasivos)
  agregar("mmargen_total", c("mactivos_margen", "mpasivos_margen"),
          quote(suma_na(mactivos_margen, mpasivos_margen)))
  # pago minimo total de las tarjetas, y medido en sueldos
  agregar("mpagominimo_total", c("Visa_mpagominimo", "Master_mpagominimo"),
          quote(suma_na(Visa_mpagominimo, Master_mpagominimo)))
  agregar("r_pagominimo_sobre_payroll", c("mpagominimo_total", "mpayroll"),
          quote(ifelse(mpayroll > 0, mpagominimo_total / mpayroll, NA_real_)))
  # la tarjeta Visa es reciente o vieja en relacion a la antiguedad del cliente
  agregar("r_visa_alta_sobre_antiguedad", c("Visa_fechaalta", "cliente_antiguedad"),
          quote(Visa_fechaalta / (cliente_antiguedad * 30)))
  # que parte de su vida lleva en el banco
  agregar("r_antiguedad_sobre_edad", c("cliente_antiguedad", "cliente_edad"),
          quote(cliente_antiguedad / (cliente_edad * 12)))
}

if (PARAM$fe$ratios || PARAM$fe$ratios2) {
  message("Ratios: creadas ", length(fe_creadas), ": ", paste(fe_creadas, collapse = ", "))
  if (length(fe_salteadas)) message("  salteadas ", length(fe_salteadas), ": ", paste(fe_salteadas, collapse = "; "))
}

# ---------- BLOQUE 1: data drifting, rankings por mes de los montos ----------
# Cada monto se reemplaza por su posicion relativa dentro del mes, con el cero fijo:
# positivos en (0, 1], negativos en [-1, 0), el cero queda en 0 y los NA quedan NA.
cols_monetarias <- character()
if (PARAM$fe$rankings) {
  rank_cero_fijo <- function(x) {
    r <- rep(NA_real_, length(x))
    pos <- which(x > 0); neg <- which(x < 0)
    r[which(x == 0)] <- 0
    if (length(pos)) r[pos] <- frank(x[pos], ties.method = "average") / length(pos)
    if (length(neg)) r[neg] <- -frank(-x[neg], ties.method = "average") / length(neg)
    r
  }
  cols_monetarias <- grep("^(m|Visa_m|Master_m)", colnames(dataset), value = TRUE)
  # paso todo a numerico antes: una columna entera truncaria los rankings a 0
  for (v in cols_monetarias) set(dataset, j = v, value = as.numeric(dataset[[v]]))
  dataset[, (cols_monetarias) := lapply(.SD, rank_cero_fijo), by = foto_mes, .SDcols = cols_monetarias]
  message("Bloque rankings: ", length(cols_monetarias), " variables de monto convertidas a ranking por mes")
}

# ---------- 6.3.1.5 FE historico: lags y deltas de orden 1 y 2 (identico al z719) ----------
# los lags usan la fila anterior de cada cliente: verifico que esten en orden cronologico
stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])

cols_lagueables <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# ---------- BLOQUE 3: tendencias en ventana, sobre las variables mas importantes ----------
cols_top <- character()
if (PARAM$fe$tendencias) {
  n <- PARAM$fe$tendencias_ventana

  # lista de variables: se fija una sola vez y queda guardada en la carpeta del experimento (reproducible)
  archivo_cols <- file.path(carpeta_exp, "tendencias_cols.txt")
  if (file.exists(archivo_cols)) {
    cols_top <- readLines(archivo_cols)
  } else {
    # si otro experimento ya fijo la lista (por ejemplo el WF7192), uso exactamente la misma
    previas <- file.path(base, paste0("WF", PARAM$comparar_con), "tendencias_cols.txt")
    previas <- previas[file.exists(previas)]
    candidatos <- file.path(base, c("WF7191/V3gan_desde202103", "WF7191_test/V3gan_desde202103",
                                    "WF7190/V3_desde202103"), "impo.txt")
    origen <- candidatos[file.exists(candidatos)][1]
    if (length(previas)) {
      file.copy(previas[length(previas)], archivo_cols)
      message("Variables para tendencias copiadas de ", previas[length(previas)])
      cols_top <- readLines(archivo_cols)
    } else {
    if (is.na(origen))
      stop("No encuentro el impo.txt del WF7191 ni del WF7190 para elegir las variables de las tendencias. ",
           "Copialo a una de esas carpetas o poné PARAM$fe$tendencias <- FALSE")
    impo <- fread(origen)                                   # viene ordenado por importancia (Gain)
    base_vars <- unique(sub("_(lag|delta)[12]$", "", impo$Feature))
    cols_top <- head(intersect(base_vars, setdiff(cols_lagueables, "kmes")), PARAM$fe$tendencias_top)
    writeLines(cols_top, archivo_cols)
    message("Variables para tendencias tomadas de ", origen)
    }
  }
  cols_top <- intersect(cols_top, cols_lagueables)
  if (!length(cols_top)) stop("La lista de variables para tendencias quedo vacia: revisar ", archivo_cols)

  # trabajo sobre una copia ordenada por cliente y mes, sin cambiar el orden de las filas del dataset
  ord <- order(dataset$numero_de_cliente, dataset$foto_mes)
  pos <- sequence(rle(dataset$numero_de_cliente[ord])$lengths)   # numero de mes dentro de cada cliente
  incompleta <- pos < n                                          # la ventana mezclaria dos clientes
  w <- (1:n) - (n + 1) / 2                                       # pesos de la pendiente de minimos cuadrados
  den <- sum(w^2)
  guardar <- function(nombre, valores) {
    valores[incompleta] <- NA
    res <- rep(NA_real_, length(ord))
    res[ord] <- valores
    set(dataset, j = nombre, value = res)
  }
  for (v in cols_top) {
    x <- as.numeric(dataset[[v]][ord])
    pendiente <- Reduce(`+`, lapply(1:n, function(j) w[j] * shift(x, n - j))) / den
    promedio <- frollmean(x, n)
    guardar(paste0(v, "_tend", n), pendiente)                    # pendiente de los ultimos n meses
    guardar(paste0(v, "_avg", n), promedio)                      # promedio
    guardar(paste0(v, "_max", n), frollapply(x, n, max))         # maximo
    guardar(paste0(v, "_min", n), frollapply(x, n, min))         # minimo
    guardar(paste0(v, "_ratioavg", n), x / promedio)             # valor actual contra su promedio
  }
  message("Bloque tendencias: ", length(cols_top), " variables x 5 = ", 5 * length(cols_top),
          " columnas nuevas (ventana de ", n, " meses)")
}

# ---------- clase binaria, azar y variables del modelo ----------
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1L, 0L)]
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]

excluir <- c("clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
if (PARAM$fe$sacar_ids)   # siguen en el dataset para los lags, los filtros y el archivo de Kaggle
  excluir <- c(excluir, "numero_de_cliente", "foto_mes", grep("^kmes", colnames(dataset), value = TRUE))
campos_buenos <- copy(setdiff(colnames(dataset), excluir))

# los cocientes pueden dar Inf o NaN (division por cero): los paso a NA, que LightGBM maneja bien
no_finitos <- 0
for (j in campos_buenos) if (is.double(dataset[[j]])) {
  malos <- which(is.nan(dataset[[j]]) | is.infinite(dataset[[j]]))
  if (length(malos)) { set(dataset, i = malos, j = j, value = NA_real_); no_finitos <- no_finitos + length(malos) }
}

# ---------- CANARITOS: seleccion de variables ----------
# Se agregan variables de puro azar (canaritos) y se entrena con los meses de training, sin mirar julio.
# Si el modelo usa una variable real menos que a los canaritos, esa variable no aporta nada y se descarta.
canaritos_out <- NULL
if (PARAM$fe$canaritos) {
  archivo_can <- file.path(carpeta_exp, paste0("canaritos_", etiqueta, ".csv"))
  if (file.exists(archivo_can)) {
    sel <- fread(archivo_can)
    if (!setequal(sel[es_canarito == FALSE, Feature], campos_buenos))
      stop("La seleccion guardada en ", archivo_can, " es de otro conjunto de variables. Cambia PARAM$experimento.")
    message("Canaritos: seleccion leida de ", archivo_can)
  } else {
    nombres_can <- paste0("canarito", seq_len(PARAM$fe$canaritos_cantidad))
    set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
    for (v in nombres_can) set(dataset, j = v, value = runif(nrow(dataset)))

    meses_can <- sort(unique(dataset$foto_mes))
    meses_can <- meses_can[meses_can >= PARAM$variantes[[1]]$desde & meses_can <= PARAM$trainingstrategy$hasta]
    filtro_can <- dataset[, foto_mes %in% meses_can]
    dcan <- lgb.Dataset(data = data.matrix(dataset[filtro_can, c(campos_buenos, nombres_can), with = FALSE]),
                        label = dataset[filtro_can, clase01], free_raw_data = TRUE)
    param_can <- copy(PARAM$lgbm$param_fijos)
    param_can$early_stopping_rounds <- NULL
    param_can$num_iterations <- PARAM$fe$canaritos_iteraciones
    semillas_can <- PARAM$semillerio$semillas[seq_len(min(PARAM$fe$canaritos_semillas, length(PARAM$semillerio$semillas)))]
    impo_can <- rbindlist(lapply(semillas_can, function(s) {
      param_can$seed <- s
      modelo <- lgb.train(data = dcan, param = param_can, verbose = -100)
      as.data.table(lgb.importance(modelo))[, .(Feature, Gain)]
    }))
    dataset[, (nombres_can) := NULL]
    rm(dcan); invisible(gc(full = TRUE, verbose = FALSE))

    # importancia promedio entre semillas; una variable que un modelo no uso cuenta como 0
    sel <- data.table(Feature = c(campos_buenos, nombres_can))
    sel <- merge(sel, impo_can[, .(gain = sum(Gain) / length(semillas_can)), by = Feature], by = "Feature", all.x = TRUE)
    sel[is.na(gain), gain := 0]
    sel[, es_canarito := Feature %in% nombres_can]
    setorder(sel, -gain)
    sel[, puesto := .I]
    umbral <- sel[es_canarito == TRUE, quantile(gain, PARAM$fe$canaritos_umbral)]
    sel[, queda := !es_canarito & gain > umbral]
    fwrite(sel, archivo_can)
  }

  n_antes <- length(campos_buenos)
  campos_buenos <- campos_buenos[campos_buenos %in% sel[queda == TRUE, Feature]]
  if (length(campos_buenos) < 20)
    stop("Los canaritos dejaron solo ", length(campos_buenos), " variables: revisar PARAM$fe$canaritos_umbral")
  can <- sel[es_canarito == TRUE]
  message("Canaritos: quedan ", length(campos_buenos), " de ", n_antes, " variables (se descartan ",
          n_antes - length(campos_buenos), "). El mejor canarito quedo en el puesto ", min(can$puesto),
          " de ", nrow(sel), "; el canarito mediano, en el ", round(median(can$puesto)))
  message("  Cuantas quedarian con otros umbrales: ",
          paste(sapply(c(0.25, 0.5, 0.75, 0.9, 1), function(q)
            paste0(q, " -> ", sel[es_canarito == FALSE & gain > quantile(can$gain, q), .N])), collapse = ";  "))
  nuevas <- intersect(fe_creadas, sel$Feature)
  if (length(nuevas))
    message("  Ratios (variable del mes): quedan ", paste(intersect(nuevas, campos_buenos), collapse = ", "),
            if (length(setdiff(nuevas, campos_buenos))) paste0(";  se descartan ", paste(setdiff(nuevas, campos_buenos), collapse = ", ")) else "")
  canaritos_out <- list(antes = n_antes, quedan = length(campos_buenos), mejor_canarito_puesto = min(can$puesto),
                        umbral = PARAM$fe$canaritos_umbral)
}

PARAM$fe$out <- list(ratios_creadas = fe_creadas, ratios_salteadas = fe_salteadas,
                     rankeadas = length(cols_monetarias), tendencias_cols = cols_top,
                     canaritos = canaritos_out, variables_modelo = length(campos_buenos))
message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ", nrow(dataset), " filas x ",
        length(campos_buenos), " variables del modelo  (", no_finitos, " valores no finitos pasados a NA)")

## Funciones

In [ ]:
# ---------- Kaggle ----------
preparar_kaggle <- function() {
  if (!file.exists("~/.kaggle/kaggle.json"))
    stop("falta ~/.kaggle/kaggle.json en la VM")
  if (Sys.which("kaggle") == "")
    invisible(system("python3 -m pip install -q --user kaggle"))
}

# sube un archivo; si falla (cupo diario, red) lo anota en pendientes_kaggle.txt para reintentar.
# los que ya subieron quedan en subidos_kaggle.txt y no se vuelven a subir al re-correr
# si una subida falla, el resto de la corrida ya no intenta: todo queda pendiente para la proxima vez
kaggle_sin_cupo <- FALSE
subir_a_kaggle <- function(archivo, mensaje) {
  archivo <- normalizePath(archivo)
  if (file.exists(file.path(carpeta_exp, "subidos_kaggle.txt")) &&
      archivo %in% readLines(file.path(carpeta_exp, "subidos_kaggle.txt"))) return(TRUE)
  if (kaggle_sin_cupo) {
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
    return(FALSE)
  }
  salida <- tryCatch(
    suppressWarnings(system2("kaggle", c("competitions", "submit", "-c", PARAM$kaggle$competencia,
                                         "-f", shQuote(archivo), "-m", shQuote(mensaje)),
                             stdout = TRUE, stderr = TRUE)),
    error = function(e) conditionMessage(e))
  ok <- any(grepl("Successfully submitted", salida))
  message("    kaggle ", if (ok) "OK   " else "ERROR", "  ", basename(archivo),
          if (ok) "" else paste0("  ", substr(paste(trimws(salida), collapse = " "), 1, 300)))
  if (ok) {
    cat(archivo, "\n", sep = "", file = file.path(carpeta_exp, "subidos_kaggle.txt"), append = TRUE)
    Sys.sleep(30)
  } else {
    kaggle_sin_cupo <<- TRUE
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
  }
  ok
}

reintentar_pendientes <- function() {
  arch <- file.path(carpeta_exp, "pendientes_kaggle.txt")
  if (!subir_kaggle || !file.exists(arch)) return(invisible())
  pendientes <- unique(setdiff(trimws(readLines(arch)), ""))
  file.remove(arch)
  message(length(pendientes), " archivos pendientes de subir a Kaggle")
  for (p in pendientes) {          # si falla uno, los siguientes quedan anotados sin intentar
    partes <- strsplit(p, "|", fixed = TRUE)[[1]]
    subir_a_kaggle(partes[1], partes[2])
  }
}

# ---------- Training strategy ----------
meses_training <- function(desde) {
  meses <- sort(unique(dataset$foto_mes), decreasing = TRUE)
  meses[meses >= desde & meses <= PARAM$trainingstrategy$hasta]
}

lgb_dataset <- function(filtro) {
  lgb.Dataset(data = data.matrix(dataset[filtro, campos_buenos, with = FALSE]),
              label = dataset[filtro, clase01], free_raw_data = TRUE)
}

# ---------- Ganancia ----------
# ordena por probabilidad, acumula la ganancia de contactar a los primeros N y devuelve el maximo
# de la curva suavizada (promedio movil de PARAM$ganancia$ventana clientes) y el corte donde ocurre
ganancia_maxima <- function(prob, gan) {
  orden <- order(prob, decreasing = TRUE)
  curva <- cumsum(gan[orden])
  suave <- frollmean(curva, PARAM$ganancia$ventana, align = "center")
  corte <- which.max(suave)
  list(ganancia = suave[corte], corte = corte)
}

# ---------- 6.3.2.2 Grid search por GANANCIA en validacion (reanudable) ----------
grid_search <- function(variante, carpeta) {
  archivo <- file.path(carpeta, "tb_grid_search.txt")
  training <- meses_training(variante$desde)
  filtro_train <- dataset[, foto_mes %in% training &
    (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]

  # combinaciones repetidas: un arbol no puede tener mas hojas que filas / min_data_in_leaf, asi que todo
  # num_leaves por encima de ese tope da exactamente el mismo modelo. Se evalua una sola de cada grupo.
  tb <- CJ(num_leaves = PARAM$grid$num_leaves, min_data_in_leaf = PARAM$grid$min_data_in_leaf)
  total <- nrow(tb)
  tb[, hojas_efectivas := pmin(num_leaves, floor(sum(filtro_train) / min_data_in_leaf))]
  tb <- tb[, .(num_leaves = min(num_leaves)), by = .(hojas_efectivas, min_data_in_leaf)][
    order(num_leaves, min_data_in_leaf), .(num_leaves, min_data_in_leaf)]
  message("  grilla: ", total, " combinaciones, ", nrow(tb), " distintas; ",
          PARAM$grid$semillas_eval, " semillas cada una")
  if (file.exists(archivo)) tb <- tb[!fread(archivo), on = .(num_leaves, min_data_in_leaf)]

  if (nrow(tb)) {
    filtro_valid <- dataset[, foto_mes %in% PARAM$trainingstrategy$validate]
    dtrain <- lgb_dataset(filtro_train)
    dvalidate <- lgb_dataset(filtro_valid)
    mvalid <- data.matrix(dataset[filtro_valid, campos_buenos, with = FALSE])
    # ganancia de contactar a cada cliente de validacion, en el mismo orden que dvalidate
    gan_valid <- dataset[filtro_valid, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]

    # metrica para el early stopping: la ganancia maxima suavizada en validacion
    feval_ganancia <- function(preds, dtrain) {
      list(name = "ganancia", value = ganancia_maxima(preds, gan_valid)$ganancia, higher_better = TRUE)
    }

    semillas_eval <- PARAM$semillerio$semillas[seq_len(PARAM$grid$semillas_eval)]
    for (i in seq_len(nrow(tb))) {
      x <- as.list(tb[i])
      evals <- rbindlist(lapply(semillas_eval, function(s) {
        param <- modifyList(PARAM$lgbm$param_fijos, c(x, list(seed = s)))
        modelo <- lgb.train(data = dtrain, valids = list(valid = dvalidate), eval = feval_ganancia,
                            param = param, verbose = -100)
        niter <- modelo$best_iter
        g <- ganancia_maxima(predict(modelo, mvalid, num_iteration = niter), gan_valid)
        rm(modelo); gc(full = TRUE, verbose = FALSE)
        data.table(semilla = s, ganancia = g$ganancia, corte = g$corte, num_iterations = niter)
      }))
      fila <- data.table(tb[i], ganancia = mean(evals$ganancia), ganancia_sd = sd(evals$ganancia),
                         corte = round(mean(evals$corte)), num_iterations = round(mean(evals$num_iterations)))
      fwrite(fila, archivo, sep = "\t", append = file.exists(archivo))
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  nl=", x$num_leaves,
              " mdil=", x$min_data_in_leaf, "  niter ", fila$num_iterations, "  corte ", fila$corte,
              "  ganancia ", round(fila$ganancia / 1e6, 2), " M")
    }
    rm(dtrain, dvalidate, mvalid); gc(full = TRUE, verbose = FALSE)
  }
  res <- fread(archivo)
  setorder(res, -ganancia)
  res
}

# ---------- 6.3.3 Final training con semillerio (reanudable: una prediccion por semilla) ----------
semillerio <- function(variante, carpeta, mejores) {
  carpeta_sem <- file.path(carpeta, "semillas")
  dir.create(carpeta_sem, showWarnings = FALSE)

  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  # modifyList: los mejores del grid PISAN a los fijos (con c() quedaban duplicados y LightGBM usaba los fijos)
  param_final <- modifyList(fijos, mejores)

  pendientes <- PARAM$semillerio$semillas[
    !file.exists(file.path(carpeta_sem, paste0("pred_s", PARAM$semillerio$semillas, ".txt")))]

  final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  if (isTRUE(PARAM$final$escalar_iteraciones)) {
    # el grid eligio las iteraciones con los meses de training; el final_train suma 202107
    factor <- dataset[foto_mes %in% final_train, .N] / dataset[foto_mes %in% meses_training(variante$desde), .N]
    param_final$num_iterations <- round(param_final$num_iterations * factor)
  }

  if (length(pendientes)) {
    dfinal_train <- lgb_dataset(dataset[, foto_mes %in% final_train])
    dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
    mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

    for (s in pendientes) {
      param_final$seed <- s
      modelo <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
      fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = predict(modelo, mfuture)),
             file.path(carpeta_sem, paste0("pred_s", s, ".txt")), sep = "\t")
      if (s == PARAM$semilla_primigenia) {          # modelo e importancia solo de la semilla primigenia
        lgb.save(modelo, file.path(carpeta, "modelo.txt"))
        fwrite(as.data.table(lgb.importance(modelo)), file.path(carpeta, "impo.txt"), sep = "\t")
      }
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  semilla ", s, " lista")
      rm(modelo); gc(full = TRUE, verbose = FALSE)
    }
    rm(dfinal_train, dfuture, mfuture); gc(full = TRUE, verbose = FALSE)
  }

  # promedio de las probabilidades de todas las semillas
  preds <- rbindlist(lapply(PARAM$semillerio$semillas, function(s)
    fread(file.path(carpeta_sem, paste0("pred_s", s, ".txt")))[, semilla := s]))
  tb_prediccion <- preds[, .(prob = mean(prob)), by = numero_de_cliente]
  fwrite(tb_prediccion, file.path(carpeta, "prediccion.txt"), sep = "\t")   # util para ensembles
  list(prediccion = tb_prediccion, param_final = param_final)
}

# ---------- Archivos para Kaggle ----------
generar_kaggle <- function(variante, carpeta, tb_prediccion) {
  dir.create(file.path(carpeta, "kaggle"), showWarnings = FALSE)
  tb <- copy(tb_prediccion)
  setorder(tb, -prob)
  for (envios in PARAM$kaggle$cortes) {
    tb[, Predicted := 0L]
    tb[1:envios, Predicted := 1L]
    archivo <- file.path(carpeta, "kaggle",
                         paste0("KA", PARAM$experimento, "_", variante$nombre, "_", envios, ".csv"))
    fwrite(tb[, .(numero_de_cliente, Predicted)], archivo, sep = ",")
    if (subir_kaggle)
      subir_a_kaggle(archivo, paste0("z719 ", variante$nombre, " semillerio=", PARAM$semillerio$cantidad,
                                     " semilla=", PARAM$semilla_primigenia, " envios=", envios))
  }
}

## Validación: los cuatro escenarios

In [ ]:
# ===== Validacion a uno y dos meses de distancia =====
# Para cada mes de prueba y cada distancia: entrena el semillerio con los 5 meses que terminan
# "distancia" meses antes del mes de prueba, predice el mes de prueba y guarda la curva de ganancia.
# Reanudable: guarda la prediccion de cada semilla en cada escenario.
variante <- PARAM$variantes[[1]]
carpeta  <- file.path(carpeta_exp, variante$nombre)
dir.create(carpeta, showWarnings = FALSE)
semillas <- PARAM$semillerio$semillas[seq_len(min(PARAM$validacion$semillas, length(PARAM$semillerio$semillas)))]

fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL
param <- modifyList(fijos, PARAM$modelo)

firma <- sub("\n$", "", as.yaml(list(modelo = PARAM$modelo, param_fijos = PARAM$lgbm$param_fijos, semillas = semillas,
  validacion = PARAM$validacion[c("meses_train")], positivos = PARAM$trainingstrategy$positivos,
  training_pct = PARAM$trainingstrategy$training_pct, variables_modelo = campos_buenos)))
archivo_firma <- file.path(carpeta, "config_corrida.yml")
if (file.exists(archivo_firma) && !identical(paste(readLines(archivo_firma), collapse = "\n"), firma))
  stop("La carpeta ", carpeta, " tiene resultados hechos con OTRA configuracion (modelo, semillas o variables). ",
       "Cambia PARAM$experimento para correr en una carpeta nueva.")
writeLines(firma, archivo_firma)

# meses de training: los 'n' meses que terminan 'distancia' meses antes del mes de prueba
todos_meses <- sort(unique(dataset$foto_mes))
meses_escenario <- function(prueba, distancia, n = PARAM$validacion$meses_train) {
  fin <- match(prueba, todos_meses) - distancia
  if (is.na(fin) || fin - n + 1 < 1) stop("No hay meses suficientes para probar ", prueba, " a ", distancia, " meses")
  todos_meses[(fin - n + 1):fin]
}
escenarios <- CJ(prueba = PARAM$validacion$meses_prueba, distancia = PARAM$validacion$distancias)
escenarios[, nombre := paste0("prueba", prueba, "_dist", distancia)]
escenarios[, meses := sapply(seq_len(.N), function(i) paste(meses_escenario(prueba[i], distancia[i]), collapse = " "))]
cat("Escenarios (", length(semillas), " semillas cada uno):\n", sep = ""); print(escenarios[, .(nombre, prueba, distancia, meses_training = meses)], row.names = FALSE)

curvas <- list(); resumen_esc <- list()
for (i in seq_len(nrow(escenarios))) {
  e <- escenarios[i]
  carpeta_esc <- file.path(carpeta, e$nombre)
  dir.create(carpeta_esc, showWarnings = FALSE)
  meses_tr <- meses_escenario(e$prueba, e$distancia)
  filtro_prueba <- dataset[, foto_mes == e$prueba]
  gan_prueba <- dataset[filtro_prueba, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]
  baja2_prueba <- dataset[filtro_prueba, as.integer(clase_ternaria == "BAJA+2")]

  pendientes <- semillas[!file.exists(file.path(carpeta_esc, paste0("pred_s", semillas, ".txt")))]
  if (length(pendientes)) {
    filtro_train <- dataset[, foto_mes %in% meses_tr &
      (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]
    dtrain <- lgb_dataset(filtro_train)
    mprueba <- data.matrix(dataset[filtro_prueba, campos_buenos, with = FALSE])
    for (s in pendientes) {
      param$seed <- s
      modelo <- lgb.train(data = dtrain, param = param, verbose = -100)
      fwrite(data.table(prob = predict(modelo, mprueba)), file.path(carpeta_esc, paste0("pred_s", s, ".txt")))
      rm(modelo); invisible(gc(full = TRUE, verbose = FALSE))
    }
    rm(dtrain, mprueba); invisible(gc(full = TRUE, verbose = FALSE))
    message(format(Sys.time(), "%a %b %d %X %Y  "), e$nombre, " listo")
  }
  probs <- sapply(semillas, function(s) fread(file.path(carpeta_esc, paste0("pred_s", s, ".txt")))$prob)
  orden <- order(rowMeans(probs), decreasing = TRUE)
  maxn <- min(3000, length(orden))
  cu <- data.table(escenario = e$nombre, prueba = e$prueba, distancia = e$distancia, envios = seq_len(maxn),
                   ganancia = (cumsum(gan_prueba[orden]) / 1e6)[seq_len(maxn)],
                   baja2_encontrados = cumsum(baja2_prueba[orden])[seq_len(maxn)])
  cu[, ganancia_suave := frollmean(ganancia, PARAM$ganancia$ventana, align = "center")]
  # tambien el modelo individual promedio, para ver si el semillerio cambia la conclusion
  ind <- rowMeans(apply(probs, 2, function(p) cumsum(gan_prueba[order(p, decreasing = TRUE)])[seq_len(maxn)] / 1e6))
  cu[, individual_suave := frollmean(ind, PARAM$ganancia$ventana, align = "center")]
  curvas[[e$nombre]] <- cu

  o <- cu[which.max(ganancia_suave)]
  meseta <- cu[ganancia_suave >= o$ganancia_suave - 1, range(envios)]
  fila <- data.table(escenario = e$nombre, prueba = e$prueba, distancia = e$distancia, baja2_mes = sum(baja2_prueba),
                     optimo = o$envios, ganancia_optimo = round(o$ganancia_suave, 2), baja2_en_optimo = o$baja2_encontrados,
                     meseta_desde = meseta[1], meseta_hasta = meseta[2],
                     optimo_individual = cu[which.max(individual_suave), envios])
  for (k in PARAM$validacion$candidatos) fila[, paste0("en_", k) := round(cu[envios == k, ganancia_suave], 2)]
  resumen_esc[[e$nombre]] <- fila
}
curvas <- rbindlist(curvas)
resumen_esc <- rbindlist(resumen_esc)
fwrite(curvas, file.path(carpeta, "validacion_curvas.csv"))
fwrite(resumen_esc, file.path(carpeta, "validacion_resumen.csv"))
message(format(Sys.time(), "%a %b %d %X %Y"), "  validacion completa")

## Resultado: ¿se corre el corte óptimo con dos meses de distancia?

In [ ]:
# ===== Resultado: ¿se corre el corte optimo con dos meses de distancia? =====
cat("Ganancia del semillerio en el mes de prueba (millones, curva suavizada)\n")
print(resumen_esc, row.names = FALSE)

# lo mismo que la entrega: dos meses de distancia, promediando los meses de prueba
cand <- PARAM$validacion$candidatos
prom <- curvas[, .(ganancia_suave = mean(ganancia_suave), baja2 = mean(baja2_encontrados)), by = .(distancia, envios)]
recomendacion <- rbindlist(lapply(sort(unique(prom$distancia)), function(d) {
  x <- prom[distancia == d & !is.na(ganancia_suave)]
  o <- x[which.max(ganancia_suave)]
  fila <- data.table(distancia = d, optimo = o$envios, ganancia_optimo = round(o$ganancia_suave, 2),
                     meseta_desde = min(x[ganancia_suave >= o$ganancia_suave - 1, envios]),
                     meseta_hasta = max(x[ganancia_suave >= o$ganancia_suave - 1, envios]))
  for (k in cand) fila[, paste0("pierde_en_", k) := round(o$ganancia_suave - x[envios == k, ganancia_suave], 2)]
  fila
}))
cat("\nPromedio de los meses de prueba. pierde_en_X = cuanto se pierde (millones) cortando en X en vez de en el optimo\n")
print(recomendacion, row.names = FALSE)
fwrite(recomendacion, file.path(carpeta, "validacion_recomendacion.csv"))

# cuantos BAJA+2 encuentra el modelo en los primeros N, a uno y a dos meses
cat("\nBAJA+2 encontrados en los primeros N clientes (promedio de los meses de prueba)\n")
print(dcast(prom[envios %in% c(500, 700, 800, 900, 1000, 1100, 1200, 1500, 2000)], envios ~ paste0("distancia_", distancia),
            value.var = "baja2", fun.aggregate = function(x) round(x[1], 1)), row.names = FALSE)

d2 <- recomendacion[distancia == max(distancia)]
d1 <- recomendacion[distancia == min(distancia)]
message("\nA ", d1$distancia, " mes: optimo en ", d1$optimo, " envios.  A ", d2$distancia, " meses (como la entrega): optimo en ",
        d2$optimo, " envios, meseta de ", d2$meseta_desde, " a ", d2$meseta_hasta, ".")
message("Corrimiento del optimo al pasar de 1 a 2 meses: ", d2$optimo - d1$optimo, " envios")

# grafico: un panel por mes de prueba, una curva por distancia
dibujar_validacion <- function() {
  meses <- sort(unique(curvas$prueba), decreasing = TRUE)
  op <- par(mfrow = c(1, length(meses) + 1), mar = c(4.2, 4.2, 3, 1)); on.exit(par(op))
  colores <- c("1" = "#1F6FB2", "2" = "#D9480F", "3" = "#2F6B2F")
  panel <- function(x, titulo) {
    x <- x[envios >= 300 & envios <= 2000 & !is.na(ganancia_suave)]
    plot(range(x$envios), range(x$ganancia_suave), type = "n", xlab = "Envios", ylab = "Ganancia (millones, suavizada)", main = titulo)
    grid(col = "grey88")
    for (d in sort(unique(x$distancia))) {
      y <- x[distancia == d]
      lines(y$envios, y$ganancia_suave, col = colores[as.character(d)], lwd = 3)
      abline(v = y[which.max(ganancia_suave), envios], col = colores[as.character(d)], lty = 2)
    }
    legend("bottomleft", bty = "n", legend = paste0(sort(unique(x$distancia)), " mes(es) de distancia"),
           col = colores[as.character(sort(unique(x$distancia)))], lwd = 3)
  }
  for (m in meses) panel(curvas[prueba == m], paste0("Prueba en ", m))
  panel(prom, "Promedio de los meses de prueba")
}
png(file.path(carpeta, "validacion_grafico.png"), width = 1900, height = 780, res = 130); dibujar_validacion(); invisible(dev.off())
dibujar_validacion()

## Paquete con los resultados para analizar
Deja `para_analizar_7199.tar.gz` en la carpeta `exp`.

In [ ]:
# ===== Paquete con los resultados para analizar =====
# Requiere las celdas 1 y 2. Junta los archivos chicos de todas las corridas en un solo comprimido.
# No incluye el dataset ni las predicciones por cliente.
experimentos <- unique(c(PARAM$comparar_con, PARAM$experimento))
patrones <- c("^tb_grid_search.*\\.txt$", "^impo\\.txt$", "^PARAM\\.yml$", "^config_corrida\\.yml$",
              "^resumen_variantes\\.csv$", "^comparacion_validacion\\.csv$", "^puntajes_kaggle\\.csv$",
              "^puntajes_ventanas\\.csv$", "^tendencias_cols\\.txt$", "^canaritos_.*\\.csv$",
              "^curva_julio\\.csv$", "^barrido_.*\\.(csv|png)$", "^semillas_.*\\.(csv|png)$", "^validacion_.*\\.(csv|png)$", "^julio_vs_public\\.csv$", "^curva_julio_vs_public\\.png$")
nombre_paquete <- paste0("para_analizar_", PARAM$experimento)
destino <- file.path(base, nombre_paquete)
unlink(destino, recursive = TRUE)
dir.create(destino, showWarnings = FALSE)
for (carp in file.path(base, paste0("WF", experimentos))) {
  encontrados <- list.files(carp, recursive = TRUE, full.names = TRUE)
  encontrados <- encontrados[grepl(paste(patrones, collapse = "|"), basename(encontrados))]
  for (f in encontrados) {
    relativo <- gsub("/", "__", substring(f, nchar(base) + 2))
    file.copy(f, file.path(destino, relativo), overwrite = TRUE)
  }
}
fwrite(dataset[, .N, by = .(foto_mes, clase_ternaria)][order(foto_mes, clase_ternaria)],
       file.path(destino, "clases_por_mes.csv"))
writeLines(colnames(dataset), file.path(destino, "columnas_dataset.txt"))
writeLines(campos_buenos, file.path(destino, "variables_modelo.txt"))
anterior <- setwd(base)
tar(paste0(nombre_paquete, ".tar.gz"), nombre_paquete, compression = "gzip")
setwd(anterior)
message("Listo: ", file.path(base, paste0(nombre_paquete, ".tar.gz")), "  (", length(list.files(destino)),
        " archivos, ", round(file.size(file.path(base, paste0(nombre_paquete, ".tar.gz"))) / 1024), " KB)")